# 2026-10-22 SOXL/SOXX 原始分钟来源与 NAS 快照刷新

只修复已结束会话的权威行情新鲜度；冻结 Stage1 策略、模型和纸面 release 均不变。
首跑在 NAS 来源注册完成后，Qlib 读取因 stdin 多进程重载失败；续跑只读已发布来源，不重复 Futu 请求。


In [ ]:
import hashlib,json,pathlib,shlex,os
import pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.data.minute_refresh import refresh_published_core
from src.data.nas_market_data import load_current_nas_binding,archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_20261022_minute_source_refresh_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(root/'src/data/minute_refresh.py')==plan['refresh_module_SHA256'] and sha(root/'src/data/nas_market_data.py')==plan['NAS_module_SHA256']
before=load_current_nas_binding(workspace_root=root)
if before['source_id']==plan['prior_NAS_source_id']:
 raw=root/'data/processed/intraday/soxl_soxx_none_1m.parquet';assert sha(raw)==plan['prior_raw_SHA256']
 for obj in before['objects']:
  p=root/obj['local_path']
  if p.exists():assert sha(p)==obj['sha256'],obj['local_path']
 state={'status':'refresh_started','plan_sha256':sha(card/'plan.json'),'refresh_calls':1,'model_fits':0,'accounts':0,'paper_release_writes':0,'orders':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 try:refreshed=refresh_published_core(root=root,session_date=plan['session_date'],start_session_date=plan['session_date'],host='127.0.0.1',port=11111)
 except Exception as exc:
  state['status']='refresh_failed';state['error_type']=type(exc).__name__;state['error']=str(exc);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');raise
 after=load_current_nas_binding(workspace_root=root)
else:
 after=before;source_manifest=json.loads((root/after['source_manifest_path']).read_text());state=json.loads((card/'run_state.json').read_text())
 assert after['as_of']==plan['session_date'] and source_manifest['parent_source_id']==plan['prior_NAS_source_id'] and state['refresh_calls']==1 and state['status']=='refresh_started'
 import exchange_calendars as xc
 from src.data.intraday import read_intraday_cache
 from src.data.intraday_quality import audit_minute_coverage
 raw=read_intraday_cache(root/'data/processed/intraday/soxl_soxx_none_1m.parquet')
 coverage=audit_minute_coverage(raw,xc.get_calendar('XNYS',start=raw.date.min(),end=plan['session_date']).schedule,symbols=['SOXL','SOXX'],timestamp_convention='right_edge',observed_at=pd.Timestamp.now(tz='UTC'))
 refreshed={'repair_requests':source_manifest['repair_requests'],'remaining_source_anomalies':coverage.loc[coverage.repair_required].to_dict('records')}
assert after['verified'] is True and after['as_of']==plan['session_date'] and after['source_id']!=plan['prior_NAS_source_id']
assert sha(root/after['source_manifest_path'])==after['source_id']
for obj in after['objects']:
 p=root/obj['local_path']
 if p.exists():assert sha(p)==obj['sha256'],obj['local_path']
manifest=json.loads((root/'data/qlib/soxl_intraday_15min/qlib_store_manifest.json').read_text());assert manifest['as_of']==plan['session_date']
qlib.init(provider_uri=str(root/'data/qlib/soxl_intraday_15min'),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2026-10-22',end_time='2026-10-22 16:00:00',freq='15min'))
flags=D.features(['SOXL','SOXX'],['$is_complete','$bar_count'],start_time='2026-10-22 09:45:00',end_time='2026-10-22 16:00:00',freq='15min').reset_index()
checks=[]
for symbol,g in flags.groupby('instrument',sort=True):
 early=g.sort_values('datetime').head(4);checks.append({'instrument':str(symbol),'first_four_rows':len(early),'complete_nonnull':int(early['$is_complete'].notna().sum()),'min_is_complete':float(early['$is_complete'].min()),'min_bar_count':float(early['$bar_count'].min())})
assert len(calendar)==26 and len(checks)==2 and all(x['first_four_rows']==4 and x['min_is_complete']==1 for x in checks)
requests=refreshed['repair_requests'];assert len(requests)<=plan['budget']['Futu_minute_requests_expected_max'] and all(x['symbol'] in ('SOXL','SOXX') for x in requests)
small={'plan_sha256':sha(card/'plan.json'),'before_source_id':before['source_id'],'after_source_id':after['source_id'],'after_as_of':after['as_of'],'after_verified_at':after['verified_at'],'NAS_object_count':len(after['objects']),'refresh_requests':requests,'remaining_source_anomalies':refreshed['remaining_source_anomalies'],'Qlib_calendar_rows':len(calendar),'Qlib_first_hour_checks':checks,'provider_manifest_sha256':sha(root/'data/qlib/soxl_intraday_15min/qlib_store_manifest.json'),'model_fits':0,'accounts':0,'paper_release_writes':0,'orders':0}
(card/'result.json').write_text(json.dumps(small,ensure_ascii=False,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_identity={'before':before['source_id'],'after':after['source_id'],'provider_uri':str(root/'data/qlib/soxl_intraday_15min')});rec.save_objects(artifact_path='analysis',source_refresh=small,Qlib_calendar=calendar,Qlib_first_hour=flags);rec.log_metrics(qlib_calendar_rows=len(calendar),source_requests=len(requests),new_models=0,new_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);remote=_ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':'Oct22_SOXL_SOXX_NAS_source_and_Qlib_minute_freshness_restored_no_forward_strategy_claim','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':remote,'verified':True},'new_model_fits':0,'new_native_accounts':0,'qualified_forward_candidate_days':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']}};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'source_id':after['source_id'],'requests':len(requests),'calendar':len(calendar),'checks':checks,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False))


## 中文结论

按冻结的一次来源刷新计划，Futu OpenD 对 SOXL/SOXX 各请求 2026-10-21—22 与 10-22 单日原始 NONE 分钟，共 **4 次**；每只基金分别返回 780 与 390 行。原始 NONE 缓存与公布调整面经现有构建流程进入 Qlib 1/5/15 分钟存储，并发布为 Yanxuan NAS 已验证快照 `bf6d5121d6cf13fb7f694e3b439d8c54ab33f61590786967b4d3f31d129f3349`，`as_of=2026-10-22`。新原始来源清单的 `parent_source_id` 为 `c7ddae9b31f9a1714d886b20625c2eec4068b9bd41801c1bac4aa7ffb5c93b6e`；NAS 全对象发布哈希核验及来源清单、两份公司行动原件限定冷读通过。

原生 Qlib `D.calendar` 在 10 月 22 日有 **26** 条15分钟日历；`D.features` 对 SOXL/SOXX 各取到26条，当日09:45—10:30前四根均 `$is_complete=1`、`$bar_count=26`。原生 Recorder `12dd136d74564851b6c17f83c9e756b1` 的来源结果、日历及特征对象已归档 NAS，SHA-256 `1e4aba62fed3c925ac53652b346be32737554ddb24b39e53604e0704453f0ae7`，三件对象限定下载与本机字节一致。2020-03-16 SOXX 仍有1条既存无效原始分钟，保持隔离；它不改变本次10月22日完整性结论。

首个 notebook 运行在 NAS 来源注册完成后，因从 `<stdin>` 运行导致 Qlib 子进程无法重载脚本而停止；后续只读已发布快照，设单进程并从脚本文件运行，**没有重复请求 Futu**，也没有新模型或资金账户。归档 Recorder 的 `before_source_id` 字段因续跑读取当前快照而误写成新ID；[身份修正](identity_correction.json)以冻结计划和新来源清单的父ID交叉证明旧ID，原 Recorder 保留不篡改。

决定 `Oct22_SOXL_SOXX_NAS_source_and_Qlib_minute_freshness_restored_no_forward_strategy_claim`。此次只修复数据新鲜度；现役纸面 release 仍有已记录的代码身份/首小时语义审核缺口，没有新候选前向日、订单或 SOXX 经济成绩。完整 Stage1 0bp 稳定超额及合同v2后续门槛仍未通过。
